Construction of ESG scores

TF/IDF as main measure, mean TF/IDF score as very good variable

Work done on E/G primary indicator words

Normalized search volume for that year is another variable. A high environmental discourse score in a year with high environmental attention could be optics or genuine concern.

How should S be introduced? A stewardship variable where firms are given better scores for consistently beating scores is very overfit. For the sake of clear-cut analysis we drop it.


In [ ]:
import yaml
import math
import polars as pl


# --------------------
# Load parquet dataset
# --------------------
df_raw = (
    pl.read_parquet("spy_10k_2015_present.parquet")
    .with_columns(
        pl.col("filing_date").dt.year().alias("year")  # What this line does: extracts filing year
    )
)
#Null checker
df_raw = df_raw.filter(
    pl.col("cik").is_not_null() &
    pl.col("gics_sector").is_not_null() &
    pl.col("year").is_not_null()
)

# --------------------
# Load ESG lexicon
# --------------------
with open("ESG_Lexicon.yml", "r") as f:
    lex = yaml.safe_load(f)

env_terms = lex["environmental"]
gov_terms = lex["governance"]

# --------------------
# Word counts
# --------------------
df_raw = df_raw.with_columns(
    pl.col("text").str.count_matches(r"(?i)\b\w+\b").alias("n_words")
)

# --------------------
# TF counts
# --------------------
df_raw = df_raw.with_columns([
    sum(pl.col("text").str.count_matches(term) for term in env_terms).alias("count_E"),
    sum(pl.col("text").str.count_matches(term) for term in gov_terms).alias("count_G")
])

# --------------------
# TF-IDF approximation
# --------------------
df_raw = df_raw.with_columns([
    ((pl.col("count_E") / pl.col("n_words")) * 1 ).alias("tfidf_E"),
    ((pl.col("count_G") / pl.col("n_words")) * 1 ).alias("tfidf_G")
])

# --------------------
# Mean TF-IDF per firm-year
# --------------------
docs_work = (
    df_raw
    .group_by(["cik", "gics_sector", "year"])
    .agg([
        pl.col("tfidf_E").mean().alias("tfidf_E"),
        pl.col("tfidf_G").mean().alias("tfidf_G")
    ])
)

print(docs_work.head(10))
print("Rows:", docs_work.height)

# --------------------
# Hardcoded annual trends
# --------------------
df_trend = pl.DataFrame({
    "year": [
        2015,2016,2017,2018,2019,
        2020,2021,2022,2023,2024
    ],
    "trend_E": [
        31.08,31.00,32.72,33.37,36.90,
        34.67,30.40,38.58,40.10,40.72
    ],
    "trend_G": [
        44.6,42.8,44.1,42.2,41.6,
        40.8,37.2,44.2,44.5,46.2
    ]
})

# --------------------
# ESG score construction
# --------------------
df_esg = (
    docs_work
    .join(df_trend, on="year", how="left")
    .with_columns([
    ((pl.col("tfidf_E") * pl.col("trend_E")) * 100).round(3).alias("E_score"),
    ((pl.col("tfidf_G") * pl.col("trend_G")) * 100).round(3).alias("G_score")
    ])
    .select([
        "cik",
        "gics_sector",
        "E_score",
        "G_score"
    ])
    .sort("cik")
)

df_esg = df_esg.with_columns([
    pl.col("E_score").round(0).cast(pl.Int64),
    pl.col("G_score").round(0).cast(pl.Int64)
])


df_esg = (
    docs_work
    .join(df_trend, on="year", how="left")
    .with_columns([
        (pl.col("tfidf_E") * pl.col("trend_E")).alias("E_score"),
        (pl.col("tfidf_G") * pl.col("trend_G")).alias("G_score")
    ])
    .select([
        "cik",
        "gics_sector",
        "year",        # must stay
        "E_score",
        "G_score"
    ])
    .sort(["cik","year"])
)







export_df = (
    df_esg
    .filter(
        pl.col("E_score").is_not_null() &
        pl.col("G_score").is_not_null()
    )
    .select([
        "cik",
        "gics_sector",
        "year",
        "E_score",
        "G_score"
    ])
)

threshold = (
    df_esg
    .filter(pl.col("E_score").is_not_null())
    .select(pl.col("E_score"))
    .sort("E_score", descending=True)
    .row(49)[0]
)

top50 = df_esg.filter(pl.col("E_score") >= threshold)

print(top50.to_pandas().to_string(index=False))

export_df = (
    df_esg
    .select([
        "cik",
        "gics_sector",
        "year",
        "E_score",
        "G_score"
    ])
)

export_df.write_csv("esg_firm_year_scores.csv")

from pathlib import Path

desktop = Path.home() / "Desktop"

export_df.write_csv(desktop / "esg_firm_year_scores.csv")

shape: (10, 5)
┌────────────┬────────────────────────┬──────┬──────────┬──────────┐
│ cik        ┆ gics_sector            ┆ year ┆ tfidf_E  ┆ tfidf_G  │
│ ---        ┆ ---                    ┆ ---  ┆ ---      ┆ ---      │
│ str        ┆ str                    ┆ i32  ┆ f64      ┆ f64      │
╞════════════╪════════════════════════╪══════╪══════════╪══════════╡
│ 0000012927 ┆ Industrials            ┆ 2023 ┆ 0.004487 ┆ 0.007023 │
│ 0001059556 ┆ Financials             ┆ 2016 ┆ 0.0      ┆ 0.009055 │
│ 0001097864 ┆ Information Technology ┆ 2025 ┆ 0.004035 ┆ 0.005681 │
│ 0001063761 ┆ Real Estate            ┆ 2021 ┆ 0.001289 ┆ 0.008853 │
│ 0000906107 ┆ Real Estate            ┆ 2020 ┆ 0.003939 ┆ 0.004516 │
│ 0001571949 ┆ Financials             ┆ 2017 ┆ 0.000668 ┆ 0.005051 │
│ 0001136893 ┆ Financials             ┆ 2017 ┆ 0.000047 ┆ 0.007248 │
│ 0001467858 ┆ Consumer Discretionary ┆ 2024 ┆ 0.010604 ┆ 0.004237 │
│ 0000277135 ┆ Industrials            ┆ 2021 ┆ 0.000961 ┆ 0.005539 │
│ 0000707549 ┆ Info

ColumnNotFoundError: unable to find column "year"; valid columns: ["cik", "gics_sector", "E_score", "G_score"]